In [4]:
import pandas as pd

df = pd.read_json('processed_df.jsonl', orient="records", lines=True)

len(df)

14455

In [1]:
import pandas as pd

df = pd.read_csv('results/relevance_summary_metrics.csv')
df.head()

,prompt_name,accuracy,f1,confusion_matrix,n_valid,n_invalid
0,binary_v1_minimal,0.763158,0.666667,"[[40, 11], [7, 18]]",76,0
1,binary_v2_anchor_descriptions,0.750000,0.666667,"[[38, 13], [6, 19]]",76,0
2,binary_v5_conservative_precision,0.776316,0.666667,"[[42, 9], [8, 17]]",76,0
3,binary_v3_event_as_news_guardrail,0.776316,0.653061,"[[43, 8], [9, 16]]",76,0
4,binary_v6_evidence_requirement,0.657895,0.638889,"[[27, 24], [2, 23]]",76,0


In [3]:
import importlib
import relevance_prompts

importlib.reload(relevance_prompts)
PROMPT_TEXT = relevance_prompts.PROMPT

# Cell 1 — SETUP ONLY for OPENAI BATCH API (binary relevance, Variant A, GPT-5) — LABEL ONLY.
# - Builds Batch JSONL requests (one per row) with custom_id=row_id
# - No API calls are executed in this cell
# - Keeps gold-label normalization helpers for later evaluation

import os, re, json, time
from dataclasses import dataclass
from typing import Optional, Dict, Any, Iterable

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROMPT_SPEC_A = {
    "name": "relevance_binary_variantA",
    "system": "You are a careful, literal classifier. Follow the rules and output format exactly. Output ONLY valid JSON.",
    "user_template": PROMPT_TEXT,  # expects {title} and {text}
}

# ---- LABELS (binary) ----
LABELS = ["IRRELEVANT", "RELEVANT"]

def normalize_gold_binary(consensus_binary) -> str:
    v = int(consensus_binary)
    return "IRRELEVANT" if v == 0 else "RELEVANT"

def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def normalize_pred_binary(label) -> Optional[str]:
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s or s in {"0", "false", "no"}:
        return "IRRELEVANT"
    if "rel" in s or s in {"1", "true", "yes"}:
        return "RELEVANT"
    return None

def _render_user_template(user_template: str, title: str, text: str) -> str:
    """
    Safe formatter:
    - Allows {title} and {text}
    - Escapes all other braces (so JSON examples in prompt don't break .format)
    """
    template = user_template.replace("{", "{{").replace("}", "}}")
    template = template.replace("{{title}}", "{title}").replace("{{text}}", "{text}")
    return template.format(title=title, text=text)

def _safe_str(x: Any) -> str:
    """Convert to a usable string; treat NaN/None as empty."""
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x)
    if s.strip().lower() in {"nan", "none"}:
        return ""
    return s

@dataclass
class BatchConfig:
    model: str = "gpt-5"
    max_output_tokens: int = 100  # label-only: keep small
    reasoning_effort: str = "minimal"  # reduce reasoning token burn
    text_verbosity: str = "low"    # encourage short output
    endpoint: str = "/v1/responses"  # Batch API will target this endpoint

# ---- Batch builder ----
def build_batch_jsonl(
    df: pd.DataFrame,
    prompt_spec: Dict[str, str],
    out_jsonl_path: str,
    cfg: Optional[BatchConfig] = None,
    row_id_col: str = "row_id",
    title_col: str = "title",
    text_col: str = "maintext",
    start: int = 0,
    stop: Optional[int] = None,
    overwrite: bool = True,
) -> str:
    """
    Writes a JSONL file for OpenAI Batch API where each line is one request:
      {"custom_id": "...", "method": "POST", "url": "/v1/responses", "body": {...}}

    Requirements:
      - prompt_spec["system"] and prompt_spec["user_template"] exist
      - user_template includes {title} and {text}
      - df has columns title_col and text_col
    """
    if cfg is None:
        cfg = BatchConfig()
    if not overwrite and os.path.exists(out_jsonl_path):
        raise FileExistsError(f"{out_jsonl_path} exists and overwrite=False")

    os.makedirs(os.path.dirname(out_jsonl_path) or ".", exist_ok=True)

    df_work = df.copy()
    if row_id_col not in df_work.columns:
        # stable row ids so you can map results back
        df_work = df_work.reset_index(drop=False).rename(columns={"index": row_id_col})

    if stop is None:
        stop = len(df_work)

    n_written = 0
    with open(out_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df_work.iloc[start:stop].iterrows():
            rid = row[row_id_col]
            title = _safe_str(row.get(title_col, ""))
            text = _safe_str(row.get(text_col, ""))

            user_msg = _render_user_template(
                prompt_spec["user_template"],
                title=title,
                text=text,
            )

            # Batch line format (one request)
            line = {
                "custom_id": str(rid),
                "method": "POST",
                "url": cfg.endpoint,
                "body": {
                    "model": cfg.model,
                    "input": [
                        {"role": "system", "content": prompt_spec["system"]},
                        {"role": "user", "content": user_msg},
                    ],
                    "max_output_tokens": cfg.max_output_tokens,
                    "reasoning": {"effort": cfg.reasoning_effort},
                    "text": {"verbosity": cfg.text_verbosity},
                },
            }

            f.write(json.dumps(line, ensure_ascii=False) + "\n")
            n_written += 1

    print(f"Wrote {n_written} requests to {out_jsonl_path}")
    return out_jsonl_path

# ---- Setup client (for later batch submit/poll/download steps) ----
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ---- Ensure binary gold labels exist (no API calls) ----
if "gold_label" not in df.columns:
    if "consensus_binary" in df.columns:
        df["gold_label"] = df["consensus_binary"].apply(normalize_gold_binary)
    # If this is your full 15k run without gold, it's fine to skip.
    # But if you're expecting gold for evaluation here, raise an error:
    # else:
    #     raise ValueError("Need df['gold_label'] or df['consensus_binary'] to create it.")


CHUNKING

In [5]:
import os, json, math
from datetime import datetime

# Assumes these already exist from your setup cell:
# - df
# - client
# - PROMPT_SPEC_A
# - BatchConfig
# - build_batch_jsonl

# -------------------------
# CONFIG (edit these only)
# -------------------------
run_name = "relevance15k"
RUN_TS = "v1"

chunk_size = 550
CHUNK_INDEX = 26  # change each run

cfg = BatchConfig(model="gpt-5", max_output_tokens=100, reasoning_effort="minimal", text_verbosity="low")

# -------------------------
# PREP
# -------------------------
os.makedirs("batches", exist_ok=True)

# Ensure stable row_id once
if "row_id" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "row_id"})

n = len(df)
n_chunks = math.ceil(n / chunk_size)

if not (0 <= CHUNK_INDEX < n_chunks):
    raise ValueError(f"CHUNK_INDEX must be in [0, {n_chunks-1}] but got {CHUNK_INDEX}")

start = CHUNK_INDEX * chunk_size
stop = min((CHUNK_INDEX + 1) * chunk_size, n)

# Only file we keep long-term:
index_path = f"batches/{run_name}_{RUN_TS}_ALL_batches.json"

# Temporary input JSONL (we'll delete after upload + batch creation)
jsonl_path = f"batches/_TMP_{run_name}_{RUN_TS}_chunk{CHUNK_INDEX:03d}_{start}_{stop}.jsonl"

# -------------------------
# Safety: prevent accidental resubmit of same (chunk_index,start,stop)
# -------------------------
if os.path.exists(index_path):
    with open(index_path, "r") as f:
        index = json.load(f)
else:
    index = []

for rec in index:
    if int(rec.get("chunk_index", -1)) == int(CHUNK_INDEX) and int(rec.get("start", -1)) == int(start) and int(rec.get("stop", -1)) == int(stop):
        raise FileExistsError(
            f"Chunk {CHUNK_INDEX} with rows [{start}:{stop}) already recorded in {index_path}.\n"
            f"Batch id: {rec.get('batch_id')}"
        )

print(f"Submitting chunk {CHUNK_INDEX+1}/{n_chunks}: rows [{start}:{stop}) (n={stop-start})")

# -------------------------
# 1) Build JSONL for this chunk (TEMP)
# -------------------------
build_batch_jsonl(
    df=df,
    prompt_spec=PROMPT_SPEC_A,
    out_jsonl_path=jsonl_path,
    cfg=cfg,
    start=start,
    stop=stop,
    overwrite=True,
)

# -------------------------
# 2) Upload JSONL
# -------------------------
with open(jsonl_path, "rb") as f:
    batch_file = client.files.create(file=f, purpose="batch")

# -------------------------
# 3) Create Batch
# -------------------------
batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
)

print("Batch created:", batch.id)

# -------------------------
# 4) Record ONLY the essentials in ALL_batches.json
# -------------------------
record = {
    "run_name": run_name,
    "run_ts": RUN_TS,
    "chunk_index": int(CHUNK_INDEX),
    "chunk_size": int(chunk_size),
    "start": int(start),
    "stop": int(stop),
    "n_rows_chunk": int(stop - start),
    "batch_id": batch.id,
    "input_file_id": batch_file.id,
    "created_at": getattr(batch, "created_at", None),
}

index.append(record)
index = sorted(index, key=lambda d: d["chunk_index"])

with open(index_path, "w") as f:
    json.dump(index, f, indent=2)

print("Updated run index:", index_path)

# -------------------------
# 5) Delete the temp JSONL (optional but recommended)
# -------------------------
try:
    os.remove(jsonl_path)
    print("Deleted temp input JSONL:", jsonl_path)
except FileNotFoundError:
    pass

print("You can disconnect now. Later: client.batches.retrieve('<batch_id>')")

Submitting chunk 27/27: rows [14300:14455) (n=155)
Wrote 155 requests to batches/_TMP_relevance15k_v1_chunk026_14300_14455.jsonl
Batch created: batch_69a5e5dd2f5481908585d5d5c19fcdb0
Updated run index: batches/relevance15k_v1_ALL_batches.json
Deleted temp input JSONL: batches/_TMP_relevance15k_v1_chunk026_14300_14455.jsonl
You can disconnect now. Later: client.batches.retrieve('<batch_id>')


In [6]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # only needed if you're using .env

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

batch_id = "batch_69a5e5dd2f5481908585d5d5c19fcdb0"
st = client.batches.retrieve(batch_id)

print("status:", st.status)
print("request_counts:", st.request_counts)
print("errors:", st.errors)
print("error_file_id:", getattr(st, "error_file_id", None))
print("output_file_id:", getattr(st, "output_file_id", None))

status: completed
request_counts: BatchRequestCounts(completed=155, failed=0, total=155)
errors: None
error_file_id: None
output_file_id: file-9p8Y1nok8g8B63uaMygzxL


In [7]:
import os, json
import pandas as pd
from typing import Optional, Dict, Any, Iterable

BATCH_ID = "batch_69a5e5dd2f5481908585d5d5c19fcdb0"

# If you truly don't want to keep raw batch output, set this to None.
# (If you keep it, it's useful for debugging later.)
OUT_PATH = "batches/chunk26.jsonl"

# Final artifact you want (row_id,label only)
FINAL_CSV_PATH = "batches/chunk26_final_labels.csv"

def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None
    
def normalize_pred_binary(label) -> Optional[str]:
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s or s in {"0", "false", "no"}:
        return "IRRELEVANT"
    if "rel" in s or s in {"1", "true", "yes"}:
        return "RELEVANT"
    return None

# ---- 1) retrieve batch ----
b = client.batches.retrieve(BATCH_ID)
print("status:", b.status)
if b.status != "completed":
    raise RuntimeError(f"Batch not completed (status={b.status}).")

output_file_id = b.output_file_id
if not output_file_id:
    raise RuntimeError("No output_file_id found on completed batch.")

print("output_file_id:", output_file_id)

# ---- 2) download output file (temporarily, then optionally delete) ----
content = client.files.content(output_file_id)
with open(OUT_PATH, "wb") as f:
    f.write(content.read())
print("saved raw batch output:", OUT_PATH)

def extract_output_text_from_batch_line(obj: dict) -> str | None:
    """
    Raw Batch output line format (Responses API):
      obj["response"]["body"]["output"] is a list containing items like:
        - {"type": "reasoning", ...}
        - {"type": "message", "content": [{"type":"output_text", "text":"..."}], ...}

    Returns assistant output text or None (e.g., if incomplete).
    """
    resp = obj.get("response") or {}
    body = resp.get("body") or {}
    output = body.get("output") or []

    if not isinstance(output, list):
        return None

    for item in output:
        if item.get("type") == "message":
            content_list = item.get("content") or []
            if not isinstance(content_list, list):
                continue
            for c in content_list:
                if isinstance(c, dict) and isinstance(c.get("text"), str):
                    return c["text"]
    return None

def load_chunk_results_fixed(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            rid = obj.get("custom_id")
            text_out = extract_output_text_from_batch_line(obj)

            parsed = extract_first_json_object(text_out) if text_out else None
            label = normalize_pred_binary(parsed.get("label")) if isinstance(parsed, dict) else None

            rows.append({"row_id": str(rid) if rid is not None else None, "label": label})

    return pd.DataFrame(rows)

# ---- 3) parse ----
preds = load_chunk_results_fixed(OUT_PATH)

print("\nLabel counts:")
print(preds["label"].value_counts(dropna=False))

# Hard fail if any missing labels (prevents silent N/As)
missing = preds["label"].isna().sum()
if missing:
    raise RuntimeError(f"Found {missing} missing labels (None/NaN). Consider rerun for those row_ids.")

print("\nRelevant share:")
print((preds["label"] == "RELEVANT").mean())

print("\nSample parsed rows:")
print(preds.head(5))

# ---- 4) write final labels CSV (row_id,label only) ----
final_labels = preds[["row_id", "label"]].copy()
final_labels = final_labels.dropna(subset=["row_id"]).drop_duplicates(subset=["row_id"], keep="last")
final_labels = final_labels.sort_values("row_id", key=lambda s: s.astype(int))

os.makedirs(os.path.dirname(FINAL_CSV_PATH) or ".", exist_ok=True)
final_labels.to_csv(FINAL_CSV_PATH, index=False)
print("\nSaved final labels CSV:", FINAL_CSV_PATH)

# ---- 5) optional: delete raw batch output to save space ----
# (Uncomment to delete)
os.remove(OUT_PATH)
print("Deleted raw batch output:", OUT_PATH)

status: completed
output_file_id: file-9p8Y1nok8g8B63uaMygzxL
saved raw batch output: batches/chunk26.jsonl

Label counts:
label
RELEVANT      123
IRRELEVANT     32
Name: count, dtype: int64

Relevant share:
0.7935483870967742

Sample parsed rows:
  row_id       label
0  14300    RELEVANT
1  14301    RELEVANT
2  14302    RELEVANT
3  14303    RELEVANT
4  14304  IRRELEVANT

Saved final labels CSV: batches/chunk26_final_labels.csv
Deleted raw batch output: batches/chunk26.jsonl


MERGING

In [3]:
import re
import glob
import pandas as pd

paths = glob.glob("batches/chunk*_final_labels.csv")

def extract_chunk_number(path):
    m = re.search(r"chunk(\d+)_", path)
    return int(m.group(1)) if m else -1

paths = sorted(paths, key=extract_chunk_number)


dfs = []
for p in paths:
    d = pd.read_csv(p, dtype={"row_id": str})
    # keep only what you expect
    d = d[["row_id", "label"]]
    d["source_file"] = p  # helpful for diagnosing duplicates
    dfs.append(d)

preds_all = pd.concat(dfs, ignore_index=True)

# 2) Basic sanity checks
print("\nTotal rows in preds_all:", len(preds_all))
print("Unique row_id count:", preds_all["row_id"].nunique())
print("\nLabel counts:")
print(preds_all["label"].value_counts(dropna=False))

# 3) Duplicate detection
dup_mask = preds_all.duplicated(subset=["row_id"], keep=False)
dups = preds_all.loc[dup_mask].sort_values(["row_id", "source_file"])

print("\nDuplicate row_id entries:", dups["row_id"].nunique())
if len(dups) > 0:
    print("Showing first 20 duplicate rows:")
    print(dups.head(20))

# Optional: keep one label per row_id (should only be needed if you truly have duplicates)
# This keeps the LAST occurrence by file order; change keep='first' if you prefer.
preds_dedup = preds_all.drop_duplicates(subset=["row_id"], keep="last").copy()

print("\nAfter de-dup:")
print("Rows:", len(preds_dedup))
print("Unique row_id:", preds_dedup["row_id"].nunique())


Total rows in preds_all: 14455
Unique row_id count: 14455

Label counts:
label
RELEVANT      9283
IRRELEVANT    5172
Name: count, dtype: int64

Duplicate row_id entries: 0

After de-dup:
Rows: 14455
Unique row_id: 14455


In [7]:
df.columns

Index(['text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count'],
      dtype='object')

In [8]:
import pandas as pd

# 1) Ensure df has row_id (only if needed)
if "row_id" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "row_id"})

# 2) Make types consistent
df["row_id"] = df["row_id"].astype(str)
preds_dedup["row_id"] = preds_dedup["row_id"].astype(str)

# 3) Keep only needed columns from preds
preds_dedup = preds_dedup[["row_id", "label"]]

# 4) Merge (left join keeps df row order/rows)
df = df.merge(preds_dedup, on="row_id", how="left")

# 5) Sanity checks
print("Missing labels after merge:", df["label"].isna().sum())
print(df["label"].value_counts(dropna=False))

Missing labels after merge: 0
label
RELEVANT      9283
IRRELEVANT    5172
Name: count, dtype: int64


In [ ]:
rel_df = df[df['label'] == "RELEVANT"]

rel_df.to_csv("relevant_samples.csv", index=False)